# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vishalseries7030/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked actions and reason codes

The action queue is a decision-support tool for prioritizing content for human review. Higher-ranked pages should have a clear, evidence-based reason for review, such as an older page with recent performance signals that merit investigation. A high score is a review priority, not a guarantee that refreshing the page will improve performance.

Each row should include its rank, content identifier, score, reason code, suggested review action, and a limitation or confidence note. The reason code should explain why the page was selected in plain language.

The top 20 items must be reviewed manually before any action is taken. Reviewers should check data completeness, business relevance, search intent, and whether a refresh is appropriate. The queue must not automatically change or publish content, and its ranking should not be described as causal evidence of refresh impact.

In [1]:

# ML-10 Section 1: Ranked actions + reason codes

from pathlib import Path
import subprocess
import numpy as np
import pandas as pd

# Locate the starter dataset
repo_dir = Path("/content/flyrank-ml-internship")
data_path = repo_dir / "data/raw/content_refresh_anonymized.csv"

if not data_path.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/vishalseries7030/flyrank-ml-internship.git",
            str(repo_dir),
        ],
        check=True,
    )

df_actions = pd.read_csv(data_path)

required = {
    "content_id", "content_type", "days_since_last_update",
    "impressions_90d", "avg_position",
}
missing = required - set(df_actions.columns)
assert not missing, f"Missing required columns: {sorted(missing)}"

# Do not use target labels or client IDs to rank actions.
queue = df_actions[
    ["content_id", "content_type", "days_since_last_update",
     "impressions_90d", "avg_position"]
].copy()

queue["days_since_last_update"] = pd.to_numeric(
    queue["days_since_last_update"], errors="coerce"
)
queue["impressions_90d"] = pd.to_numeric(
    queue["impressions_90d"], errors="coerce"
)
queue["avg_position"] = pd.to_numeric(
    queue["avg_position"], errors="coerce"
)

queue["stale_flag"] = (
    queue["days_since_last_update"] >= 180
).astype(int)

queue["visible_flag"] = (
    queue["impressions_90d"] >= 300
).astype(int)

# Position 0 means no position data, not rank zero.
queue["position_opportunity_flag"] = (
    queue["avg_position"].between(11, 50)
).astype(int)

# Transparent heuristic: visibility × simple review-priority multipliers.
# This is a prioritization rule, not a trained probability.
queue["action_score"] = (
    np.log1p(queue["impressions_90d"].clip(lower=0).fillna(0))
    * (
        1
        + 2 * queue["stale_flag"]
        + queue["position_opportunity_flag"]
    )
)

def assign_reason(row):
    if row["stale_flag"] and row["visible_flag"]:
        return "stale_and_visible"
    if row["position_opportunity_flag"] and row["visible_flag"]:
        return "visible_position_opportunity"
    if row["visible_flag"]:
        return "high_visibility_review"
    return "lower_signal_manual_review"

queue["reason_code"] = queue.apply(assign_reason, axis=1)

queue["suggested_action"] = queue["reason_code"].map({
    "stale_and_visible": "Review freshness and update accuracy",
    "visible_position_opportunity": "Review search intent and on-page relevance",
    "high_visibility_review": "Inspect current content and performance",
    "lower_signal_manual_review": "Check data quality before prioritizing",
})

queue["review_note"] = (
    "Human review required; score does not predict refresh impact"
)

# Rank deterministically and inspect the first 20 items.
queue = queue.sort_values(
    ["action_score", "content_id"],
    ascending=[False, True],
).reset_index(drop=True)

queue.insert(0, "rank", np.arange(1, len(queue) + 1))

top20 = queue.head(20).copy()

display(
    top20[
        [
            "rank", "content_id", "content_type",
            "days_since_last_update", "impressions_90d",
            "avg_position", "action_score", "reason_code",
            "suggested_action", "review_note",
        ]
    ].round({"action_score": 3})
)

print("Dataset rows:", len(df_actions))
print("Ranked queue rows:", len(queue))
print("Top-20 rows:", len(top20))
print("Target labels used for ranking: No")
print("Client IDs used for ranking: No")

assert len(queue) == len(df_actions)
assert len(top20) == min(20, len(df_actions))
assert queue["content_id"].is_unique
assert queue["action_score"].notna().all()

print("PASS: ranked queue generated with reason codes.")
print("Note: thresholds and heuristic weights are review rules, not validated causal effects.")


,rank,content_id,content_type,days_since_last_update,impressions_90d,avg_position,action_score,reason_code,suggested_action,review_note
0,1,content_cf56e2e2e282,keyword article,194,61678,19.7,44.119,stale_and_visible,Review freshness and update accuracy,Human review required; score does not predict ...
1,2,content_7368877ea310,keyword article,194,59472,24.8,43.973,stale_and_visible,Review freshness and update accuracy,Human review required; score does not predict ...
2,3,content_1bfaa38ff26c,keyword article,194,25715,22.2,40.619,stale_and_visible,Review freshness and update accuracy,Human review required; score does not predict ...
3,4,content_5feee3994adb,keyword article,194,7812,39.0,35.854,stale_and_visible,Review freshness and update accuracy,Human review required; score does not predict ...
4,5,content_c2d929d83eaa,keyword article,193,7558,17.9,35.722,stale_and_visible,Review freshness and update accuracy,Human review required; score does not predict ...
5,6,content_b16bd7307b39,keyword article,194,4590,31.0,33.727,stale_and_visible,Review freshness and update accuracy,Human review required; score does not predict ...
6,7,content_fe16a55cd13d,keyword article,194,4556,16.4,33.698,stale_and_visible,Review freshness and update accuracy,Human review required; score does not predict ...
7,8,content_ecb6215e79fd,keyword article,194,4429,25.3,33.585,stale_and_visible,Review freshness and update accuracy,Human review required; score does not predict ...
8,9,content_928af3e22c80,keyword article,193,1697,15.8,29.749,stale_and_visible,Review freshness and update accuracy,Human review required; score does not predict ...
9,10,content_bdbec75c1148,keyword article,194,1316,21.8,28.732,stale_and_visible,Review freshness and update accuracy,Human review required; score does not predict ...


Dataset rows: 30000
Ranked queue rows: 30000
Top-20 rows: 20
Target labels used for ranking: No
Client IDs used for ranking: No
PASS: ranked queue generated with reason codes.
Note: thresholds and heuristic weights are review rules, not validated causal effects.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use and limits

The Content Action Playbook is intended to help an editor or SEO analyst prioritize pages for manual review. It uses a transparent heuristic score based on page freshness, impressions, and a search-position opportunity flag. Reason codes explain why a page was placed in the queue.

The score is a prioritization aid, not a probability of decline and not a prediction that refreshing a page will improve performance. Thresholds and weights are initial review rules and have not been validated as optimal.

The current dataset contains 30,000 rows, and the generated queue ranks those rows without using target labels or client IDs. The earlier ML-08 evaluation was performed on a client-grouped test set of 6,163 keyword articles from seven held-out clients. That evaluation does not validate this separate heuristic queue, and its results should not be generalized to other content types.

The queue should be used only to recommend review. Editors must check data quality, search intent, content accuracy, business priorities, and whether an update is appropriate. No content should be automatically changed or published based on this score alone.

In [2]:

# ML-10 Section 2: Intended-use and limits checks

assert len(queue) == len(df_actions)
assert queue["action_score"].notna().all()
assert queue["reason_code"].notna().all()
assert "target_declining_proxy" not in queue.columns
assert "trend_direction" not in queue.columns
assert "client_id" not in queue.columns

print("Intended user: editor or SEO analyst")
print("Purpose: prioritize pages for human review")
print("Queue type: heuristic, not an ML probability")
print("Automatic publishing: not permitted")
print("Rows ranked:", len(queue))
print("Unique reason codes:", queue["reason_code"].nunique())
print("PASS: intended-use boundaries checked.")


Intended user: editor or SEO analyst
Purpose: prioritize pages for human review
Queue type: heuristic, not an ML probability
Automatic publishing: not permitted
Rows ranked: 30000
Unique reason codes: 4
PASS: intended-use boundaries checked.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review and the no-go list

Every recommended page must be reviewed by a person before any change is made. The reviewer should confirm that the page identifier and metrics are correct, the content is still relevant, the search intent is understood, and the proposed update would provide genuine value to readers.

The reviewer should also check for missing or stale analytics, recent changes that may not be reflected in the data, brand and factual accuracy, legal or compliance requirements, and whether the page has a valid reason to be refreshed. The reason code is a starting point for investigation, not a final decision.

**No-go list:**
- Never automatically rewrite, update, delete, or publish a page based only on the score.
- Never claim that the score proves a refresh will increase traffic or rankings.
- Never expose private client information or sensitive queries in exports.
- Never prioritize a page without checking data quality and business context.
- Never treat missing metrics as evidence that a page is performing poorly.
- Never bypass editorial, legal, accessibility, or subject-matter review.

The system should produce recommendations for human decision-making only. Reviewers should be able to reject a recommendation and record the reason.

In [3]:

# ML-10 Section 3: Human review and no-go checks

review_checklist = [
    "Verify page identity and metric accuracy",
    "Check content freshness and search intent",
    "Confirm the proposed update adds reader value",
    "Check missing or stale analytics",
    "Review factual, brand, legal, and accessibility risks",
]

no_go_rules = [
    "No automatic content edits or publishing",
    "No causal claims based on the score",
    "No private client names or sensitive queries in exports",
    "No action based only on missing metrics",
    "Human reviewer can reject any recommendation",
]

print("Human review checklist:")
for item in review_checklist:
    print("-", item)

print("\nNo-go rules:")
for rule in no_go_rules:
    print("-", rule)

assert len(review_checklist) >= 5
assert len(no_go_rules) >= 5
assert any("automatic" in rule.lower() for rule in no_go_rules)
assert any("reject" in rule.lower() for rule in no_go_rules)

print("\nPASS: human-review checklist and no-go rules defined.")


Human review checklist:
- Verify page identity and metric accuracy
- Check content freshness and search intent
- Confirm the proposed update adds reader value
- Check missing or stale analytics
- Review factual, brand, legal, and accessibility risks

No-go rules:
- No automatic content edits or publishing
- No causal claims based on the score
- No private client names or sensitive queries in exports
- No action based only on missing metrics
- Human reviewer can reject any recommendation

PASS: human-review checklist and no-go rules defined.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring and retraining triggers

The action queue should be reviewed monthly and whenever the source data or scoring logic changes. Monitoring should track data freshness, missing-value rates, score distributions, reason-code frequencies, and how often editors accept or reject recommendations.

A review should be triggered if important input fields become unavailable, missingness increases sharply, the score distribution changes substantially, or the top-20 queue becomes unexpectedly different from the previous run. These are proposed operational thresholds and should be calibrated using observed history.

Once outcomes from human-reviewed recommendations become available, evaluate whether the queue helps editors prioritize useful work. Measure subsequent performance over a defined future window and compare it with a suitable baseline. Do not assume that a refresh caused an improvement without a suitable comparison design.

Retraining is not automatically required for this heuristic queue. Revisit its rules and weights when monitoring shows sustained changes, and validate any revised scoring approach before replacing the current version. Keep a record of each run, rule version, review decision, and subsequent outcome.

In [5]:

# ML-10 Section 4: Monitoring and review triggers

monitoring = {
    "run_frequency": "Monthly and after material data/rule changes",
    "missingness_review_trigger": 0.10,  # Proposed 10% threshold
    "top20_overlap_review_trigger": 0.50,  # Review if overlap falls below 50%
    "score_drift_review_trigger": 0.20,  # Review if median score changes by >20%
}

# Check source-data fields in df_actions.
source_columns = [
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
]

missing_rates = (
    df_actions[source_columns]
    .isna()
    .mean()
    .sort_values(ascending=False)
)

# action_score is created in queue, not in df_actions.
median_score = float(queue["action_score"].median())

print("Missing-value rates:")
display(missing_rates.rename("missing_rate").to_frame().round(3))

print("\nCurrent median action score:", round(median_score, 3))

print("\nProposed monitoring triggers:")
for name, value in monitoring.items():
    print(f"{name}: {value}")

triggered_missing = missing_rates[
    missing_rates > monitoring["missingness_review_trigger"]
]

print("\nFields exceeding proposed 10% missingness threshold:")
if triggered_missing.empty:
    print("None")
else:
    display(triggered_missing.rename("missing_rate").to_frame().round(3))

assert queue["action_score"].notna().all()
assert median_score >= 0
assert 0 < monitoring["missingness_review_trigger"] < 1
assert 0 < monitoring["top20_overlap_review_trigger"] < 1

print("\nPASS: current queue health checked and monitoring triggers defined.")
print("Note: score drift and top-20 overlap need a saved prior run for comparison.")


Missing-value rates:


,missing_rate
days_since_last_update,0.0
impressions_90d,0.0
avg_position,0.0



Current median action score: 8.641

Proposed monitoring triggers:
run_frequency: Monthly and after material data/rule changes
missingness_review_trigger: 0.1
top20_overlap_review_trigger: 0.5
score_drift_review_trigger: 0.2

Fields exceeding proposed 10% missingness threshold:
None

PASS: current queue health checked and monitoring triggers defined.
Note: score drift and top-20 overlap need a saved prior run for comparison.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*


### Exports for the Paper

Export the ranked action queue as a CSV for review and reproducibility. The export includes content identifiers, content type, ranking score, reason code, suggested action, and the human-review limitation.

The export must not include client names, private URLs, sensitive queries, or internal client identifiers. Before sharing, verify that the output contains only approved columns and that the queue is ranked correctly.

The CSV is a review artifact, not evidence that refreshing content will improve performance. All recommendations require human review before action.


In [6]:

# ML-10 Section 5: Export queue and run final checks

from pathlib import Path

# Export only the approved, non-sensitive columns.
export_columns = [
    "rank",
    "content_id",
    "content_type",
    "action_score",
    "reason_code",
    "suggested_action",
    "review_note",
]

missing_columns = [
    col for col in export_columns if col not in queue.columns
]
assert not missing_columns, f"Missing export columns: {missing_columns}"

export_df = queue[export_columns].copy()

# Confirm the export is complete and correctly ranked.
assert len(export_df) == len(queue) == 30000
assert export_df["rank"].is_unique
assert export_df["rank"].is_monotonic_increasing
assert export_df["content_id"].notna().all()
assert export_df["action_score"].notna().all()
assert export_df["reason_code"].notna().all()

# Exclude potentially sensitive fields.
forbidden_columns = {"client_id", "client_name", "url", "query"}
assert not forbidden_columns.intersection(export_df.columns)

# Save the CSV beside the notebook.
notebook_dir = Path.cwd()
if notebook_dir.name != "notebooks":
    candidate = notebook_dir / "work" / "notebooks"
    if candidate.exists():
        notebook_dir = candidate

output_path = notebook_dir / "ml10_ranked_action_queue.csv"
export_df.to_csv(output_path, index=False)

print("Export saved to:", output_path)
print("Export rows:", len(export_df))
print("Export columns:", export_df.columns.tolist())
print("\nTop 5 rows:")
display(export_df.head())

print("\nPASS: export created and final checks passed.")
print("Reminder: human review required; score does not establish causal impact.")


Export saved to: /content/ml10_ranked_action_queue.csv
Export rows: 30000
Export columns: ['rank', 'content_id', 'content_type', 'action_score', 'reason_code', 'suggested_action', 'review_note']

Top 5 rows:


,rank,content_id,content_type,action_score,reason_code,suggested_action,review_note
0,1,content_cf56e2e2e282,keyword article,44.118795,stale_and_visible,Review freshness and update accuracy,Human review required; score does not predict ...
1,2,content_7368877ea310,keyword article,43.973111,stale_and_visible,Review freshness and update accuracy,Human review required; score does not predict ...
2,3,content_1bfaa38ff26c,keyword article,40.619475,stale_and_visible,Review freshness and update accuracy,Human review required; score does not predict ...
3,4,content_5feee3994adb,keyword article,35.854177,stale_and_visible,Review freshness and update accuracy,Human review required; score does not predict ...
4,5,content_c2d929d83eaa,keyword article,35.721977,stale_and_visible,Review freshness and update accuracy,Human review required; score does not predict ...



PASS: export created and final checks passed.
Reminder: human review required; score does not establish causal impact.


In [7]:

from pathlib import Path
import shutil

repo = Path("/content/flyrank-ml-internship")
assert repo.exists(), "Repository folder not found"

source = Path("/content/ml10_ranked_action_queue.csv")
destination = repo / "work" / "notebooks" / "ml10_ranked_action_queue.csv"

destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(source, destination)

print("CSV copied to:", destination)
print("File exists:", destination.exists())


CSV copied to: /content/flyrank-ml-internship/work/notebooks/ml10_ranked_action_queue.csv
File exists: True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.